In [ ]:
from pathlib import Path

import pandas as pd

# Sets the path to the parent directory of RR classes
base_dir = Path.cwd().parent
data_dir = base_dir / "Data"

# Import data from the O*NET database, at ISCO-08 occupation level.
# The original data uses a version of SOC classification, but the data we load here
# are already cross-walked to ISCO-08 using: https://ibs.org.pl/en/resources/occupation-classifications-crosswalks-from-onet-soc-to-isco/

# The O*NET database contains information for occupations in the USA, including
# the tasks and activities typically associated with a specific occupation.

task_data = pd.read_csv(data_dir / "onet_tasks.csv")
# isco08 variable is for occupation codes
# the t_* variables are specific tasks conducted on the job

# read employment data from Eurostat
# These datasets include quarterly information on the number of workers in specific
# 1-digit ISCO occupation categories. (Check here for details: https://www.ilo.org/public/english/bureau/stat/isco/isco08/)

employment_file = data_dir / "Eurostat_employment_isco.xlsx"
isco_data = {
    isco: pd.read_excel(employment_file, sheet_name=f"ISCO{isco}")
    for isco in range(1, 10)
}

isco1 = isco_data[1]
isco2 = isco_data[2]
isco3 = isco_data[3]
isco4 = isco_data[4]
isco5 = isco_data[5]
isco6 = isco_data[6]
isco7 = isco_data[7]
isco8 = isco_data[8]
isco9 = isco_data[9]

selected_countries = ["Belgium", "Spain", "Poland"]

# This will calculate worker totals in each of the chosen countries.
country_totals = {
    country: sum(isco_frame[country] for isco_frame in isco_data.values())
    for country in selected_countries
}
total_Belgium = country_totals["Belgium"]
total_Spain = country_totals["Spain"]
total_Poland = country_totals["Poland"]

In [ ]:
import pandas as pd

# Let's merge all these datasets. We'll need a column that stores the occupation categories:
for isco, isco_frame in isco_data.items():
    isco_frame["ISCO"] = isco

# and this gives us one large file with employment in all occupations.
all_data = pd.concat(list(isco_data.values()), ignore_index=True)

# We have 9 occupations and the same time range for each, so we can add the totals by
# adding a vector that is 9 times the previously calculated totals
for country, total_series in country_totals.items():
    all_data[f"total_{country}"] = pd.concat(
        [total_series] * len(isco_data), ignore_index=True
    )

# And this will give us shares of each occupation among all workers in a period-country
for country in selected_countries:
    all_data[f"share_{country}"] = all_data[country] / all_data[f"total_{country}"]

In [ ]:
# Now let's look at the task data. We want the first digit of the ISCO variable only
import pandas as pd
import numpy as np

task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[:1].astype(int)

# And we'll calculate the mean task values at a 1-digit level
# (more on what these tasks are below)
aggdata = task_data.groupby(["isco08_1dig"], as_index=False).mean(numeric_only=True)
aggdata = aggdata.drop(columns=["isco08"])

# We'll be interested in tracking the intensity of Non-routine cognitive analytical tasks
# Using a framework reminiscent of the work by David Autor.

# These are the ones we're interested in:
# Non-routine cognitive analytical
# 4.A.2.a.4 Analyzing Data or Information
# 4.A.2.b.2 Thinking Creatively
# 4.A.4.a.1 Interpreting the Meaning of Information for Others

task_columns = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]

# Let's combine the data.
combined = pd.merge(
    all_data, aggdata, left_on="ISCO", right_on="isco08_1dig", how="left"
)
# Traditionally, the first step is to standardise the task values using weights
# defined by share of occupations in the labour force. This should be done separately
# for each country. Standardisation -> getting the mean to 0 and std. dev. to 1.
# Let's do this for each of the variables that interests us:

In [ ]:
def weighted_standardize(values, weights):
    temp_mean = np.average(values, weights=weights)
    temp_sd = np.sqrt(np.average((values - temp_mean) ** 2, weights=weights))
    return (values - temp_mean) / temp_sd


country_order = ["Belgium", "Poland", "Spain"]

# first task item
for country in country_order:
    combined[f"std_{country}_t_4A2a4"] = weighted_standardize(
        combined["t_4A2a4"],
        combined[f"share_{country}"],
    )

# second task item
for country in country_order:
    combined[f"std_{country}_t_4A2b2"] = weighted_standardize(
        combined["t_4A2b2"],
        combined[f"share_{country}"],
    )

# third task item
for country in country_order:
    combined[f"std_{country}_t_4A4a1"] = weighted_standardize(
        combined["t_4A4a1"],
        combined[f"share_{country}"],
    )

In [ ]:
# The next step is to calculate the `classic` task content intensity, i.e.
# how important is a particular general task content category in the workforce
# Here, we're looking at non-routine cognitive analytical tasks, as defined
# by David Autor and Darron Acemoglu:

import matplotlib.pyplot as plt

combined["Belgium_NRCA"] = (
    combined["std_Belgium_t_4A2a4"]
    + combined["std_Belgium_t_4A2b2"]
    + combined["std_Belgium_t_4A4a1"]
)
combined["Poland_NRCA"] = (
    combined["std_Poland_t_4A2a4"]
    + combined["std_Poland_t_4A2b2"]
    + combined["std_Poland_t_4A4a1"]
)
combined["Spain_NRCA"] = (
    combined["std_Spain_t_4A2a4"]
    + combined["std_Spain_t_4A2b2"]
    + combined["std_Spain_t_4A4a1"]
)

# And we standardise NRCA in a similar way.
for country in country_order:
    combined[f"std_{country}_NRCA"] = weighted_standardize(
        combined[f"{country}_NRCA"],
        combined[f"share_{country}"],
    )


# Finally, to track the changes over time, we have to calculate a country-level mean
# Step 1: multiply the value by the share of such workers.
for country in country_order:
    combined[f"multip_{country}_NRCA"] = (
        combined[f"std_{country}_NRCA"] * combined[f"share_{country}"]
    )

# Step 2: sum it up (it basically becomes another weighted mean)
agg_Spain = combined.groupby(["TIME"], as_index=False)["multip_Spain_NRCA"].sum()
agg_Belgium = combined.groupby(["TIME"], as_index=False)["multip_Belgium_NRCA"].sum()
agg_Poland = combined.groupby(["TIME"], as_index=False)["multip_Poland_NRCA"].sum()

# We can plot it now!

plt.plot(agg_Poland["TIME"], agg_Poland["multip_Poland_NRCA"])
plt.xticks(range(0, len(agg_Poland), 3), agg_Poland["TIME"][::3])
plt.show()

plt.plot(agg_Spain["TIME"], agg_Spain["multip_Spain_NRCA"])
plt.xticks(range(0, len(agg_Spain), 3), agg_Spain["TIME"][::3])
plt.show()

plt.plot(agg_Belgium["TIME"], agg_Belgium["multip_Belgium_NRCA"])
plt.xticks(range(0, len(agg_Belgium), 3), agg_Belgium["TIME"][::3])
plt.show()

# To run the analysis for another country, add that country name to
# `selected_countries` in the first cell, then create the corresponding
# country-specific NRCA, weighted mean, aggregation, and plot lines by
# following the Belgium, Poland, and Spain examples in this cell.
# E.g. for Finland:
# total_Finland = country_totals["Finland"]
# all_data["share_Finland"] = all_data["Finland"] / all_data["total_Finland"]
# combined["std_Finland_t_4A2a4"] = weighted_standardize(combined["t_4A2a4"], combined["share_Finland"])
# combined["std_Finland_t_4A2b2"] = weighted_standardize(combined["t_4A2b2"], combined["share_Finland"])
# combined["std_Finland_t_4A4a1"] = weighted_standardize(combined["t_4A4a1"], combined["share_Finland"])
# combined["Finland_NRCA"] = combined["std_Finland_t_4A2a4"] + combined["std_Finland_t_4A2b2"] + combined["std_Finland_t_4A4a1"]
# combined["std_Finland_NRCA"] = weighted_standardize(combined["Finland_NRCA"], combined["share_Finland"])
# combined["multip_Finland_NRCA"] = combined["std_Finland_NRCA"] * combined["share_Finland"]
# agg_Finland = combined.groupby(["TIME"], as_index=False)["multip_Finland_NRCA"].sum()
# plt.plot(agg_Finland["TIME"], agg_Finland["multip_Finland_NRCA"])
# plt.xticks(range(0, len(agg_Finland), 3), agg_Finland["TIME"][::3])
# plt.show()

# Routine manual
# 4.A.3.a.3	Controlling Machines and Processes
# 4.C.2.d.1.i	Spend Time Making Repetitive Motions
# 4.C.3.d.3	Pace Determined by Speed of Equipment